# Introduction to Arabic Speech Technologies
## Chapter 4 notebook: A hidden Markov model with two states and three frames

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

Chapter 4 explains that the Viterbi algorithm scores the single best alignment between frames and HMM states, while the forward algorithm sums over every alignment, and notes that the two answers differ. This notebook carries out both calculations on the smallest example that shows the difference: two emitting states and three acoustic frames. The second half turns the same machinery on a small left-to-right model for بَاب /baːb/, the word used in Chapter 4.

**Contents**

1. The model: states, transitions and Gaussian emissions
2. The forward algorithm: summing over all alignments
3. Viterbi: the single best alignment
4. Every path, written out, as a check
5. Working in the log domain
6. A left-to-right model for بَاب

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/notebooks/ch04_hmm_forward_and_viterbi.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 1. The model

Two emitting states, each with a one-dimensional Gaussian emission density, a left-to-right transition matrix that allows a self-loop or a move to the next state, and three observed frames.

In [ ]:
import numpy as np
from itertools import product

pi = np.array([1.0, 0.0])                 # start in state 1
A  = np.array([[0.7, 0.3],                # self-loop, move on
               [0.0, 1.0]])
means, sds = np.array([0.0, 2.0]), np.array([1.0, 1.0])
frames = np.array([0.2, 1.1, 2.3])        # three acoustic frames, one dimension each

def gauss(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))

B = np.array([[gauss(f, means[s], sds[s]) for s in range(2)] for f in frames])
print("emission likelihoods b_s(x_t), rows = frames, columns = states")
print(np.round(B, 4))

## 2. The forward algorithm

$\alpha_t(s)$ is the probability of the frames seen so far together with being in state $s$ at time $t$. Summing $\alpha_T$ over the states gives the total likelihood of the observation sequence under the model, over every alignment.

In [ ]:
alpha = np.zeros((len(frames), 2))
alpha[0] = pi * B[0]
for t in range(1, len(frames)):
    for s in range(2):
        alpha[t, s] = B[t, s] * np.sum(alpha[t - 1] * A[:, s])
total = alpha[-1].sum()
print("alpha:\n", np.round(alpha, 6))
print(f"\ntotal likelihood (forward) = {total:.8f}")

## 3. Viterbi

The same recursion with the sum replaced by a maximum, plus a backtrace that recovers the best state sequence.

In [ ]:
delta = np.zeros((len(frames), 2)); psi = np.zeros((len(frames), 2), dtype=int)
delta[0] = pi * B[0]
for t in range(1, len(frames)):
    for s in range(2):
        scores = delta[t - 1] * A[:, s]
        psi[t, s] = int(np.argmax(scores))
        delta[t, s] = B[t, s] * np.max(scores)
best_end = int(np.argmax(delta[-1])); best = [best_end]
for t in range(len(frames) - 1, 0, -1):
    best.append(psi[t, best[-1]])
best.reverse()
viterbi = delta[-1].max()
print("delta:\n", np.round(delta, 6))
print(f"\nbest path likelihood (Viterbi) = {viterbi:.8f}")
print("best state sequence (1-indexed):", [int(s) + 1 for s in best])
print(f"\nthe best single path accounts for {100*viterbi/total:.1f}% of the total likelihood")

## 4. Every path, written out

With two states and three frames there are only a handful of legal paths, so the two answers above can be checked directly.

In [ ]:
rows = []
for path in product(range(2), repeat=len(frames)):
    p = pi[path[0]] * B[0, path[0]]
    for t in range(1, len(frames)):
        p *= A[path[t - 1], path[t]] * B[t, path[t]]
    if p > 0:
        rows.append((tuple(s + 1 for s in path), p))
rows.sort(key=lambda r: -r[1])
print(f"{'path':12} {'probability':>14}")
for path, p in rows:
    print(f"{str(path):12} {p:14.8f}")
print(f"{'sum':12} {sum(p for _, p in rows):14.8f}  (forward: {total:.8f})")
print(f"{'max':12} {max(p for _, p in rows):14.8f}  (Viterbi: {viterbi:.8f})")

## 5. The log domain

Real utterances have hundreds of frames, and a product of hundreds of small numbers underflows. Recognisers therefore work with log probabilities, replacing multiplication by addition and the sum by a numerically safe log-sum-exp.

In [ ]:
def logsumexp(v):
    m = np.max(v)
    return m + np.log(np.sum(np.exp(v - m)))

log_pi, log_A, log_B = np.log(pi + 1e-300), np.log(A + 1e-300), np.log(B + 1e-300)
log_alpha = np.zeros((len(frames), 2))
log_alpha[0] = log_pi + log_B[0]
for t in range(1, len(frames)):
    for s in range(2):
        log_alpha[t, s] = log_B[t, s] + logsumexp(log_alpha[t - 1] + log_A[:, s])
print(f"log total likelihood = {logsumexp(log_alpha[-1]):.6f}  (direct: {np.log(total):.6f})")

## 6. A left-to-right model for بَاب

Chapter 4 draws a three-phone model for بَاب /baːb/ with three states per phone. The cell builds that model with placeholder Gaussians and decodes a synthetic feature sequence, printing the state alignment and the implied phone boundaries. The point is the shape of the alignment, not the acoustic realism.

In [ ]:
phones = ["b", "aː", "b"]
states = [f"{p}{i+1}" for p in phones for i in range(3)]      # three states per phone
n = len(states)
centres = np.array([0.0, 0.1, 0.2, 1.8, 2.0, 1.9, 0.2, 0.1, 0.0])

P = np.zeros((n, n))
for i in range(n):
    P[i, i] = 0.6
    if i + 1 < n: P[i, i + 1] = 0.4
P[-1, -1] = 1.0

rng = np.random.default_rng(3)
true_durations = [4, 3, 3, 6, 7, 5, 3, 4, 5]
obs = np.concatenate([centres[i] + 0.25 * rng.standard_normal(d) for i, d in enumerate(true_durations)])

logB = np.array([[np.log(gauss(o, c, 0.45) + 1e-300) for c in centres] for o in obs])
logP = np.log(P + 1e-300)
d = np.full((len(obs), n), -np.inf); bp = np.zeros((len(obs), n), dtype=int)
d[0, 0] = logB[0, 0]
for t in range(1, len(obs)):
    for s in range(n):
        prev = d[t - 1] + logP[:, s]
        bp[t, s] = int(np.argmax(prev)); d[t, s] = logB[t, s] + np.max(prev)
path = [int(np.argmax(d[-1]))]
for t in range(len(obs) - 1, 0, -1):
    path.append(bp[t, path[-1]])
path.reverse()

print("frames:", len(obs))
print("state path:", " ".join(states[s] for s in path))
boundaries = [(states[path[i]][:-1], i) for i in range(1, len(path)) if states[path[i]][:-1] != states[path[i-1]][:-1]]
print("\nphone boundaries (frame index):", boundaries)
print("frame 10 ms apart, so /aː/ starts at", f"{boundaries[0][1]*0.01:.2f} s")